# Mixture Models and EM

In [ ]:
%load_ext autotime
%load_ext nb_black
%matplotlib inline

import os
from collections import defaultdict
import torch
import numpy as np
import pandas as pd
import cvxpy as cp
import scipy.stats
from scipy import optimize
from sympy import *
from torch.distributions import constraints
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 300
plt.rcParams["figure.figsize"] = (16, 12)

import pyro
import pyro.distributions as dist
from pyro import poutine
from pyro.infer.autoguide import AutoDelta
from pyro.optim import Adam
from pyro.infer import SVI, TraceEnum_ELBO, config_enumerate, infer_discrete

smoke_test = "CI" in os.environ
assert pyro.__version__.startswith("1.5.1")
pyro.enable_validation(True)

## $K$-means

### Problem Setup

- I.I.D. Observed data: $\mathcal{X} = \left(\mathbf{x}^{(1)}, \mathbf{x}^{(2)}, \cdots, \mathbf{x}^{(N)}\right), \mathbf{x}^{(i)} \in \mathbb{R}^{D}$
    
- Goal: Assign each observation $\mathbf{x}^{(i)}$ to a cluster $k = 1, \cdots, K$ 

- How?

    - Find a set of vectors representing each cluster $(\mu_1, \cdots, \mu_K)$
    
    - Assign each observation to the cluster $k$ if $k = \underset{j}{\arg\min} \lVert \mathbf{x}^{(i)} - \mu_j \rVert^2 $

### Optimization Formulation

\begin{aligned}
    \underset{\mu_j, r_{ik} \forall i \in \{1, \cdots, N\}, k \in \{1, \cdots, K \}}{\arg\min} &\sum^{N}_{i=1}\sum^{K}_{j=1}r_{ik}\lVert \mathbf{x}^{(i)} - \mu_j \rVert^2 \\
    \text{subject to }
    r_{ik} &= 
    \begin{cases}
        1 \\
        0 \\
    \end{cases} \\
\end{aligned}